# Imports

In [ ]:
import os
import sys
import pandas as pd

# 1. Path Setup for Docker/Local consistency
# We need to reach the project root to see the 'utils' folder.
current_dir = os.getcwd()
if os.path.basename(current_dir) == 'data_science':
    project_root = os.path.abspath(os.path.join(current_dir, '..'))
else:
    project_root = current_dir

if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Import Global Styling
try:
    from utils import apply_base_style, CUSTOM_PALETTE
    apply_base_style()
except ImportError as e:
    print(f"❌ Error importing utils: {e}")
    print(f"Project root: {project_root}")

# 3. Ensure working directory is correct for data loading
if os.path.basename(os.getcwd()) != 'data_science' and os.path.exists('data_science'):
    os.chdir('data_science')

print(f"Working Directory: {os.getcwd()}")

# Configuration
Simple check to verify the file is where we think it is.


In [ ]:
# Define relative paths
INPUT_FILE_ARTICLES = "../data_processed/articles.json"
INPUT_FILE_BLOG_POSTS = "../data_processed/blog_posts.json"
INPUT_FILE_EVENTS = "../data_processed/events.json"

# Simple check to verify the files are where we think they are
if os.path.exists(INPUT_FILE_ARTICLES):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_ARTICLES}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_ARTICLES}")

if os.path.exists(INPUT_FILE_BLOG_POSTS):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_BLOG_POSTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_BLOG_POSTS}")

if os.path.exists(INPUT_FILE_EVENTS):
    print(f"✅ Setup complete. Input file found: {INPUT_FILE_EVENTS}")
else:
    print(f"❌ Warning: Input file NOT found at {INPUT_FILE_EVENTS}")

# Load raw data
These are processed data, ready for analyses. They are stored in JSON format within the `data_processed/` folder (*or dummy examples* inside `data_processed_dummy/`).

### Load articles data from a JSON file
This content is similar to blog posts; the primary difference is the ***storage format***.  
- ***Blog posts*** are a classical blog post data, while 
- ***articles*** are text created within Wix CMS table, and has no metrics (detecting *likes* and *views* for these cases require a workaround, which is time consuming).

In [ ]:
try:
    # 1. Load the data
    _articles_raw = pd.read_json(INPUT_FILE_ARTICLES)
    print(f"✅ Loaded {len(_articles_raw)} Article records.")
    
    # 2. Output results
    display(_articles_raw.head(2))

except Exception as e:
    print(f"❌ Error loading articles: {e}")

#### The main "Source of Truth" for Articles Data Analysis

In [ ]:
articles_df = _articles_raw.copy() 
print("🎯 The main 'Source of Truth' for Articles Data Analysis")
print("\n📑 Available columns:", articles_df.columns.tolist())
display(articles_df.head(2))

### Load blog posts data from a JSON file

These are classical blog data, containing `metrics` (`likes`, `views`). 

In [ ]:
try:
    # 1. Load the data
    _blog_posts_raw = pd.read_json(INPUT_FILE_BLOG_POSTS)
    print(f"✅ Loaded {len(_blog_posts_raw)} Blog Post records.")
    
    # 2. Output results
    display(_blog_posts_raw.head(2))

except Exception as e:
    print(f"❌ Error loading blog posts: {e}")

#### The main "Source of Truth" for Blog Posts Data Analysis & Flattening `metrics`

In [ ]:
# 1. Create a fresh copy to work with
blog_posts_df = _blog_posts_raw.copy()

# 2. Flatten the metrics
# .fillna({}) ensures that if 'metrics' is None/NaN, it becomes an empty dict instead of crashing
metrics_flat = pd.json_normalize(blog_posts_df['metrics'].fillna({}))

# 3. Join back to the main dataframe
blog_posts_df = pd.concat([
    blog_posts_df.drop(columns=['metrics']).reset_index(drop=True),
    metrics_flat.reset_index(drop=True)
], axis=1)

# 4. Handle missing values in the new columns
# This turns NaN into 0 for the metrics columns
metric_cols = metrics_flat.columns
blog_posts_df[metric_cols] = blog_posts_df[metric_cols].fillna(0)

# 5. Output results
print("🎯 The main 'Source of Truth' for Blog Posts Data Analysis\n")
print(f"📑 Available columns: {blog_posts_df.columns.tolist()}")
display(blog_posts_df.head(2))

### Load events data from a JSON file

In [ ]:
try:
    # 1. Load the data
    _events_raw = pd.read_json(INPUT_FILE_EVENTS)
    print(f"✅ Loaded {len(_events_raw)} Event records.")

    # 2. Output results
    display(_events_raw.head(2))

except Exception as e:
    print(f"❌ Error loading events: {e}")

####The main "Source of Truth" for Events Data Analysis & Flattening `eventGuests`

In [ ]:
# 1. Create a fresh copy to work with
events_df = _events_raw.copy()

# 2. Flatten the eventGuests
# .fillna({}) ensures that if 'eventGuests' is None/NaN, it becomes an empty dict instead of crashing
event_guests_flat = pd.json_normalize(events_df['eventGuests'].fillna({})).add_prefix('guest.')

# 3. Join back to the main dataframe
events_df = pd.concat([
    events_df.drop(columns=['eventGuests']).reset_index(drop=True),
    event_guests_flat.reset_index(drop=True)
], axis=1)

# 4. Handle missing values in the new columns
# This turns NaN into 0 for the eventGuests columns
metric_cols = event_guests_flat.columns
events_df[metric_cols] = events_df[metric_cols].fillna(0)

# 5. Output results
print("🎯 The main 'Source of Truth' for Events Data Analysis\n")
print(f"📑 Available columns: {events_df.columns.tolist()}")
display(events_df.head(2))  

# Convert date string to actual datetime objects

[NOTE]   
While `pd.read_json()` often auto-detects date strings, these columns are explicitly converted to ensure they are typed as `datetime64[ns]`.   
This guarantees consistency across different data sources and provides the necessary foundation for the temporal trend analysis.

In [ ]:
def convert_and_verify(df, col_name, label):
    print(f"--- {label} ---")

    # Check if it's already a datetime object
    if pd.api.types.is_datetime64_any_dtype(df[col_name]):
           print("💡 Note: Pandas already auto-converted this during loading!")

    print(f"Current format in memory: {df[col_name].iloc[0]}")
    print(f"Dtype: {df[col_name].dtype}\n")

    # We still run this to be safe (it won't hurt if already converted)
    df[col_name] = pd.to_datetime(df[col_name], errors='coerce')

convert_and_verify(articles_df, 'publishedDate', 'Articles')
convert_and_verify(blog_posts_df, 'publishedDate', 'Blog Posts')
convert_and_verify(events_df, 'date', 'Events')

# Plotting Table - Combined Data

In [ ]:
# 1. Prepare Articles 
articles_temp_df = articles_df[['publishedDate', 'title', 'category']].copy()
articles_temp_df['Type'] = 'Article'
articles_temp_df = articles_temp_df.rename(columns={'publishedDate': 'Date', 'category': 'Category',
'title': 'Title'})

# Fill metrics with 0 (since they aren't available)
for col in ['Blog_Views', 'Blog_Likes', 'Guest_Total', 'Guest_Going', 'Guest_Waitlist']:
    articles_temp_df[col] = 0

# 2. Prepare Blog Posts 
blog_temp_df = blog_posts_df[['publishedDate', 'title', 'categories', 'views', 'likes']].copy()
blog_temp_df['Type'] = 'Blog Post'
blog_temp_df = blog_temp_df.rename(columns={'publishedDate': 'Date', 'categories': 'Category', 'title': 'Title', 'views':
'Blog_Views', 'likes': 'Blog_Likes'})

# Fill event-specific metrics with 0
for col in ['Guest_Total', 'Guest_Going', 'Guest_Waitlist']:
    blog_temp_df[col] = 0

# 3. Prepare Events
events_temp_df = events_df[['date', 'title', 'category', 'guest.total', 'guest.going',
'guest.waitlist']].copy()
events_temp_df['Type'] = 'Event'
events_temp_df = events_temp_df.rename(columns={
    'date': 'Date',
    'category': 'Category',
    'title': 'Title',
    'guest.total': 'Guest_Total',
    'guest.going': 'Guest_Going',
    'guest.waitlist': 'Guest_Waitlist'
})

#  Fill blog-specific metrics with 0
for col in ['Blog_Views', 'Blog_Likes']:
    events_temp_df[col] = 0

# 4. Combine everything
combined_df = pd.concat([articles_temp_df, blog_temp_df, events_temp_df], ignore_index=True)


# 5. Feature Engineering
combined_df['Year'] = combined_df['Date'].dt.year
combined_df['Month'] = combined_df['Date'].dt.month

# Final Check
print("The Combined DataFrame\n")
print(f"📑 Available columns: {combined_df.columns.tolist()}")
display(combined_df.sample(15))

[NOTE]  
`Category` column  is a mix of `strings` (from *Articles*/*Events*) and `lists of strings` (from *Blog Posts*).  

Why this is happening:    
- ***Articles*** & ***Events***: These came from a source where an item belongs to a single category (e.g., "*simontonklub*").  
- ***Blog Posts***: These allow for multiple tags, so the data is stored as a Python list (e.g., ['*#ÉldMegAPillanatot*', '*#Vendégblog*']).  

How to fix it:     
- Flatten to a String  


Sample from the  Combined DataFrame:

| ID  | Date       | Title                                                         | Category                                   | Type      | Blog_Views | Blog_Likes | Guest_Total | Guest_Going | Guest_Waitlist | Year | Month |
|-----|------------|---------------------------------------------------------------|--------------------------------------------|-----------|------------|------------|-------------|-------------|----------------|------|-------|
| 223 | 2021-03-09 | Meditáció, mint gyógyító erő - Összehangolva t...            | meditációgyógyítóerő                       | Event     | 0          | 0          | 17          | 14          | 2              | 2021 | 3     |
| 145 | 2024-10-25 | Összehangolva a Városligetben - SIMONTON KLUB                | simontonklub                               | Event     | 0          | 0          | 33          | 23          | 10             | 2024 | 10    |
| 250 | 2020-05-19 | ÚJ IDŐPONT NYÍLT! Online Erőforrás Tréning 6+1...             | erőforrásnap                               | Event     | 0          | 0          | 13          | 13          | 0              | 2020 | 5     |
| 112 | 2020-03-01 | Szegény gazdagok – avagy a marokkói esztergályos             | [#VeddÉszre, #ÉldMegAPillanatot]            | Blog Post | 173        | 5          | 0           | 0           | 0              | 2020 | 3     |
| 86  | 2021-03-03 | A "VAN" ereje                                                | [#VeddÉszre]                                | Blog Post | 536        | 7          | 0           | 0           | 0              | 2021 | 3     |
| 91  | 2020-12-23 | Szenteste "meséje"                                           | [#ÉldMegAPillanatot, #ÉletMesék]            | Blog Post | 342        | 10         | 0           | 0           | 0              | 2020 | 12    |
| 248 | 2020-07-09 | Összehangolva a Simonton módszerrel – segítség...            | simontonösszehangolva                      | Event     | 0          | 0          | 22          | 14          | 8              | 2020 | 7     |



# The main "Source of Truth" for Combined Data Analysis 

In [ ]:
combined_df = combined_df.copy()

# Convert lists to comma-separated strings; keep existing strings as they are
combined_df['Category'] = combined_df['Category'].apply(
    lambda x: ', '.join(x) if isinstance(x, list) else x
)

print("🎯 The main 'Source of Truth' for Combined Data Analysis")
display(combined_df.sample(5))